# Atlas integrated scoring

This notebook calculates Levels 1 and 2 of the **Atlas Integrated Structure** for every BSR represented in the five input CSVs.

| Level | Main question | Principal outputs |
|---|---|---|
| **1. Integrated risk** | Where do fish use, limiting-factor condition, biological vulnerability, and population priority overlap? | Life-stage impact and risk, limiting-factor impact and risk, and overall BSR impact and risk |
| **2. Action benefit** | Which action types address the limiting factors contributing most to calculated risk? | Condition improvement, limiting-factor amelioration, and overall action benefit by BSR and action type |

The Life-Stage Fish Use Score (`LS_corrected_score`) is specific to the BSR, species, and life stage and is the fish-use multiplier in both impact and risk. It is retained on its source scale and is not constrained to 0 to 1. Values in the current input data range from 0 to 3.59. The species-level `species_aggregate_score` and BSR-level `fish_use_score_decimal` are retained for reporting and quality control but are not direct multipliers.

The notebook recalculates limiting-factor condition from its source 1-to-5 rating to a 0.01-to-1.0 scale and vulnerability from rank 1-to-15 to a 1.0-to-0.01 scale. The 0.10-to-1.0 condition and vulnerability scores supplied in the CSVs are retained as source fields but are not used in the equations. The LFAT score equals relationship directness multiplied by frequency. Level 3 project scoring is not included.

The seven core score CSVs and `bsr_scores.gpkg` are written to `data/outputs`. Supporting review and quality-control tables are written to `data/outputs/QC`.


## 1. Inputs, outputs, and the only routine setting

Place the following files in `data/inputs`:

- `Fish Use Scores.csv`
- `LFAT.csv`
- `Limiting factor scores.csv`
- `Population scores.csv`
- `Vulnerability table.csv`
- `bsr.gpkg`

The automatic file search also accepts a single parenthetically numbered copy of each input, such as `Fish Use Scores(8).csv` or `bsr(1).gpkg`. If needed, set `INPUT_DIR_OVERRIDE` below to the folder containing the inputs. No other settings are normally required.


In [8]:
from contextlib import closing
from pathlib import Path
from uuid import uuid4
import hashlib
import shutil
import sqlite3

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


# Optional: replace None with a folder path if automatic discovery is not appropriate.
INPUT_DIR_OVERRIDE = None

INPUT_STEMS = {
    "fish_use": "Fish Use Scores",
    "lfat": "LFAT",
    "limiting_factor": "Limiting factor scores",
    "population": "Population scores",
    "vulnerability": "Vulnerability table",
}
FISH_USE_COLUMNS = [
    "bsr", "basin", "bsr_crosswalk_status",
    "species", "life_stage", "LS_corrected_score",
    "species_aggregate_score", "fish_use_score_decimal",
]
BSR_INPUT_FILE = "bsr.gpkg"
BSR_OUTPUT_FILE = "bsr_scores.gpkg"


def select_input_file(folder, stem, suffix=".csv"):
    # Return one exact or parenthetically numbered input, or None.
    exact = folder / f"{stem}{suffix}"
    if exact.exists():
        return exact
    matches = sorted(folder.glob(f"{stem}(*){suffix}"))
    return matches[0] if len(matches) == 1 else None


def candidate_input_directories(start):
    seen = set()
    for folder in (start, *start.parents):
        for candidate in (
            folder / "data" / "inputs",
            folder / "upload",
            folder,
        ):
            resolved = candidate.resolve()
            if resolved not in seen:
                seen.add(resolved)
                yield resolved


def locate_inputs(start, override=None):
    candidates = [Path(override).expanduser().resolve()] if override else list(
        candidate_input_directories(start)
    )
    for folder in candidates:
        if not folder.is_dir():
            continue
        selected = {
            key: select_input_file(folder, stem)
            for key, stem in INPUT_STEMS.items()
        }
        spatial_path = select_input_file(folder, "bsr", ".gpkg")
        if (
            all(path is not None for path in selected.values())
            and spatial_path is not None
        ):
            return folder, selected, spatial_path
    expected = [
        *(f"{stem}.csv" for stem in INPUT_STEMS.values()),
        BSR_INPUT_FILE,
    ]
    raise FileNotFoundError(
        "Could not find one complete input set. Expected: "
        + ", ".join(expected)
    )


INPUT_DIR, INPUT_PATHS, BSR_INPUT_PATH = locate_inputs(
    Path.cwd().resolve(), INPUT_DIR_OVERRIDE
)
if INPUT_DIR.name == "inputs" and INPUT_DIR.parent.name == "data":
    REPO_ROOT = INPUT_DIR.parent.parent
else:
    REPO_ROOT = Path.cwd().resolve()

OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
QC_DIR = OUTPUT_DIR / "QC"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
BSR_OUTPUT_PATH = OUTPUT_DIR / BSR_OUTPUT_FILE

raw = {
    key: pd.read_csv(
        path, usecols=FISH_USE_COLUMNS if key == "fish_use" else None
    )
    for key, path in INPUT_PATHS.items()
}

input_summary = pd.DataFrame(
    [
        {
            "dataset": key,
            "file": path.name,
            "rows": len(raw[key]),
            "columns": len(raw[key].columns),
        }
        for key, path in INPUT_PATHS.items()
    ]
)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"QC directory: {QC_DIR}")
print(f"Spatial input: {BSR_INPUT_PATH.name}")
display(input_summary)


Input directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\inputs
Output directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\outputs
QC directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\outputs\QC
Spatial input: bsr.gpkg


,dataset,file,rows,columns
0,fish_use,Fish Use Scores.csv,290,8
1,lfat,LFAT.csv,165,19
2,limiting_factor,Limiting factor scores.csv,435,18
3,population,Population scores.csv,20,4
4,vulnerability,Vulnerability table.csv,180,14


## 2. Score fields and transformations

### Fish-use inputs

- **Life-Stage Fish Use Score** is specific to the BSR, species, and life stage. It is the fish-use multiplier used in both the impact and risk calculations.
- **Life-Stage Fish Use Score** is the **Corrected\_{life stage}** score in the `Normalized Score Calculator` tab and the `LS_corrected_score` field in the fish-use input table.
- The Life-Stage Fish Use Score is retained on its source scale. It is not constrained to 0 to 1. Values in the current input data range from 0 to 3.59.
- **Species Fish Use Score** is the species-specific score listed in the `Fish Use Scores - Normalized` tab and the `species_aggregate_score` field. It is reported for context but is not a direct multiplier in the impact or risk equations.
- **Overall Fish Use Score** is the BSR-level 0-to-1 score in the `fish_use_score_decimal` field. It is reported for context but is not a direct multiplier in the impact or risk equations.

### Vulnerability inputs

- **Vulnerability Rank** is specific to the species, life stage, and limiting factor. Rank 1 represents the highest vulnerability and rank 15 represents the lowest vulnerability.
- The vulnerability score used in the calculations maps rank 1 to 1.0 and rank 15 to 0.01:

$$
\text{vulnerability}
=
1
-
\left(
\text{vulnerability rank} - 1
\right)
\times
\frac{0.99}{14}
$$

- The provided `vulnerability_score` field uses a previous 0.10-to-1.0 transformation. It is retained for reference, but the calculations use the score recalculated from `vulnerability_rank` with the equation above.
- For the combined Migration life stage, the calculation uses the higher of the adult and juvenile vulnerability scores for each species and limiting factor.

### Limiting-factor inputs

- **Limiting-Factor Condition** is specific to the BSR and limiting factor. The current input table does not contain a life-stage-specific condition field.
- The condition score used in the calculations maps the source 1-to-5 rating to 0.01 to 1.0:

$$
\text{limiting-factor condition}
=
0.01
+
\left(
\text{raw condition rating} - 1
\right)
\times
\frac{0.99}{4}
$$

- The provided `lf_condition_score` field uses a previous 0.10-to-1.0 transformation. It is retained for reference, but the calculations use the score recalculated from `lf_condition_score_raw_1_5` with the equation above.
- Within a BSR, the same condition score for a limiting factor is applied to every species and life-stage pathway.

### Population-priority inputs

- **Population Priority** is specific to the basin, species, and life stage.
- Population priority is used only in the risk calculation. It is not used in the impact calculation.
- Population-priority values sum to approximately 1 within each basin and species. Small differences from 1 can result from rounding in the source table.


In [9]:
EXPECTED_COLUMNS = {
    "fish_use": FISH_USE_COLUMNS,
    "lfat": [
        "action_id", "action_type", "action_definition",
        "source_action_label", "limiting_factor",
        "limiting_factor_occurrence", "directness_code",
        "directness_rating", "directness_value", "frequency_code",
        "frequency_rating", "frequency_value", "lfat_score",
        "source_sheet", "source_row", "source_directness_cell",
        "source_frequency_cell", "source_score_cell", "source_notes_cell",
    ],
    "limiting_factor": [
        "bsr", "limiting_factor", "n", "mean_r", "median_r",
        "geo_mean_r", "sd_r", "var_r", "iqr_r", "min_r", "max_r",
        "range_r", "flag_spread", "flag_low_n", "any_flag",
        "lf_condition_score_raw_1_5", "lf_condition_score",
        "condition_transformation",
    ],
    "population": [
        "basin", "species", "life_stage", "population_priority",
    ],
    "vulnerability": [
        "species", "source_life_stage", "life_stage", "limiting_factor",
        "vulnerability_rank", "vulnerability_score", "uncertainty_flag",
        "review_flag", "review_reason", "source_sheet",
        "source_rank_cell", "source_rating_cell", "source_notes_cell",
        "source_uncertainty_cell",
    ],
}

# This fixed list lets later checks confirm complete 15-factor coverage.
CANONICAL_LF = [
    "Anthropogenic Barriers",
    "Riparian Condition",
    "Floodplain Condition",
    "Side Channel and Wetland Habitat",
    "Channel and Habitat Structure",
    "Decreased Water Quantity",
    "Altered Flow Timing",
    "Decreased Sediment Quantity",
    "Increased Sediment Quantity",
    "Summer Water Temperature",
    "Winter Water Temperature",
    "Water Quality",
    "Predation",
    "Altered Primary Productivity",
    "Non-Native Species Interactions and Competition",
]

# Compare the actual columns in each input with the required columns above.
schema_rows = []
for key, table in raw.items():
    observed = table.columns.tolist()
    expected = EXPECTED_COLUMNS[key]
    schema_rows.append(
        {
            "dataset": key,
            "observed_columns": len(observed),
            "expected_columns": len(expected),
            "schema_pass": observed == expected,
        }
    )
schema_qc = pd.DataFrame(schema_rows)

# Stop if an input has missing, extra, or reordered columns.
if not schema_qc["schema_pass"].all():
    details = {
        key: {
            "observed": raw[key].columns.tolist(),
            "expected": EXPECTED_COLUMNS[key],
        }
        for key in raw
        if raw[key].columns.tolist() != EXPECTED_COLUMNS[key]
    }
    raise ValueError(f"One or more input schemas do not match: {details}")

# Rename BSR fish use for stable output naming. It is retained for context only.
fish_use = raw["fish_use"].rename(
    columns={"fish_use_score_decimal": "fish_use_score"}
).copy()

# Copy population priorities so the imported table is not modified.
population = raw["population"].copy()

# Preserve the CSV's condition score and give the raw rating a concise name.
condition = raw["limiting_factor"].rename(
    columns={
        "lf_condition_score_raw_1_5": "condition_score_raw_1_5",
        "lf_condition_score": "condition_score_source",
    }
).copy()

# Convert condition linearly: source rating 1 becomes 0.01 and 5 becomes 1.0.
condition["condition_score"] = (
    0.01
    + (condition["condition_score_raw_1_5"] - 1.0) * (0.99 / 4.0)
)

# Preserve the CSV's vulnerability score before recalculating it from rank.
vulnerability = raw["vulnerability"].rename(
    columns={"vulnerability_score": "vulnerability_score_source"}
).copy()

# Convert vulnerability linearly: rank 1 becomes 1.0 and rank 15 becomes 0.01.
vulnerability["vulnerability_score"] = (
    1.0
    - (vulnerability["vulnerability_rank"] - 1.0) * (0.99 / 14.0)
)

# Copy the action crosswalk so derived calculations do not modify the import.
lfat = raw["lfat"].copy()

# Record the implemented definitions for the saved QC documentation.
assumptions = pd.DataFrame(
    [
        ["Life-stage fish use", "LS_corrected_score", "Used in both impact and risk."],
        ["Species fish use", "species_aggregate_score", "Context only; not a direct multiplier."],
        ["BSR fish use", "fish_use_score_decimal", "Context only; not a direct multiplier."],
        ["Population priority", "population_priority", "Risk equals impact multiplied by priority."],
        ["Limiting-factor condition", "condition_score", "Raw 1 maps to 0.01 and 5 maps to 1.0."],
        ["Vulnerability", "vulnerability_score", "Rank 1 maps to 1.0 and 15 maps to 0.01."],
        ["Combined migration", "maximum vulnerability_score", "Uses the higher adult or juvenile value."],
        ["LFAT", "lfat_score", "Directness multiplied by frequency."],
        ["BSR identifiers", "bsr", "Fish-use and condition files use the same identifiers."],
    ],
    columns=["component", "field_or_rule", "implementation"],
)

display(schema_qc)
display(assumptions)


,dataset,observed_columns,expected_columns,schema_pass
0,fish_use,8,8,True
1,lfat,19,19,True
2,limiting_factor,18,18,True
3,population,4,4,True
4,vulnerability,14,14,True


,component,field_or_rule,implementation
0,Life-stage fish use,LS_corrected_score,Used in both impact and risk.
1,Species fish use,species_aggregate_score,Context only; not a direct multiplier.
2,BSR fish use,fish_use_score_decimal,Context only; not a direct multiplier.
3,Population priority,population_priority,Risk equals impact multiplied by priority.
4,Limiting-factor condition,condition_score,Raw 1 maps to 0.01 and 5 maps to 1.0.
5,Vulnerability,vulnerability_score,Rank 1 maps to 1.0 and 15 maps to 0.01.
6,Combined migration,maximum vulnerability_score,Uses the higher adult or juvenile value.
7,LFAT,lfat_score,Directness multiplied by frequency.
8,BSR identifiers,bsr,Fish-use and condition files use the same iden...


## 3. Align vulnerability life stages

The vulnerability table contains separate `Adult Migration & Holding` and `Juvenile Emigration` rows, while the fish-use and population tables contain one combined `Migration` life stage. The notebook collapses the two source rows to one species × life stage × limiting-factor relationship.

$$
\begin{aligned}
\text{combined migration vulnerability}
={}&
\max\Bigl(
\text{adult migration vulnerability},\\
&\qquad \text{juvenile migration vulnerability}
\Bigr)
\end{aligned}
$$

The calculation is applied separately for each species and limiting factor using the recalculated 0.01-to-1.0 vulnerability score. Taking the maximum represents the more vulnerable migration pathway without counting migration twice. Other life stages pass through unchanged.


In [10]:
def any_yes(values):
    # Replace missing values with blanks and convert all values to text.
    cleaned_values = values.fillna("").astype(str)

    # Ignore surrounding spaces and capitalization when checking for "yes".
    is_yes = cleaned_values.str.strip().str.lower().eq("yes")

    # Return one flag for the full group.
    return "Yes" if is_yes.any() else "No"


# Group by the keys used in scoring. Adult and juvenile migration share the
# combined "Migration" life_stage value and therefore enter the same group.
vulnerability_collapsed = (
    vulnerability.groupby(
        ["species", "life_stage", "limiting_factor"], as_index=False
    )
    .agg(
        # Retain source rank and recalculated-score ranges for review.
        vulnerability_rank_min=("vulnerability_rank", "min"),
        vulnerability_rank_max=("vulnerability_rank", "max"),
        vulnerability_score_min=("vulnerability_score", "min"),
        vulnerability_score_max=("vulnerability_score", "max"),

        # Use the larger adult/juvenile score in the scoring equations.
        vulnerability_score=("vulnerability_score", "max"),

        # Document how many and which source stages contributed.
        source_vulnerability_rows=("source_life_stage", "size"),
        source_life_stages=(
            "source_life_stage",
            lambda values: " | ".join(sorted(set(values.astype(str)))),
        ),

        # Combine review flags and uncertainty notes for the group.
        vulnerability_review_flag=("review_flag", any_yes),
        uncertainty_notes=(
            "uncertainty_flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

display(vulnerability_collapsed.head(10))


,species,life_stage,limiting_factor,vulnerability_rank_min,vulnerability_rank_max,vulnerability_score_min,vulnerability_score_max,vulnerability_score,source_vulnerability_rows,source_life_stages,vulnerability_review_flag,uncertainty_notes
0,Bull Trout,FMO,Altered Flow Timing,11,11,0.292857,0.292857,0.292857,1,FMO (Fluvial),Yes,Unknown Future Climate Change Impacts
1,Bull Trout,FMO,Altered Primary Productivity,12,12,0.222143,0.222143,0.222143,1,FMO (Fluvial),No,
2,Bull Trout,FMO,Anthropogenic Barriers,3,3,0.858571,0.858571,0.858571,1,FMO (Fluvial),No,
3,Bull Trout,FMO,Channel and Habitat Structure,4,4,0.787857,0.787857,0.787857,1,FMO (Fluvial),No,
4,Bull Trout,FMO,Decreased Sediment Quantity,13,13,0.151429,0.151429,0.151429,1,FMO (Fluvial),Yes,Don't know how sediment quantity impacts FMO
5,Bull Trout,FMO,Decreased Water Quantity,2,2,0.929286,0.929286,0.929286,1,FMO (Fluvial),Yes,Unknown Future Climate Change Impacts
6,Bull Trout,FMO,Floodplain Condition,9,9,0.434286,0.434286,0.434286,1,FMO (Fluvial),No,
7,Bull Trout,FMO,Increased Sediment Quantity,14,14,0.080714,0.080714,0.080714,1,FMO (Fluvial),No,
8,Bull Trout,FMO,Non-Native Species Interactions and Competition,10,10,0.363571,0.363571,0.363571,1,FMO (Fluvial),No,
9,Bull Trout,FMO,Predation,7,7,0.575714,0.575714,0.575714,1,FMO (Fluvial),Yes,We don't know the extent of predation. Would b...


## 4. Calculate Level 1 pathway impact and risk

### Pathway components

For each BSR, species, life stage, and limiting factor combination:

$$
\text{impact component}
=
\text{life-stage fish use}
\times
\text{limiting-factor condition}
\times
\text{vulnerability}
$$

$$
\text{risk component}
=
\text{impact component}
\times
\text{population priority}
$$

Vulnerability and population priority are specific to the applicable species and life stage. Limiting-factor condition is specific to the BSR and limiting factor and is applied to all species and life-stage pathways for that limiting factor within the BSR.

Impact and risk can be aggregated in two equivalent ways:

1. Across the 15 limiting factors for each species and life stage.
2. Across all species and life stages for each limiting factor.

Both approaches produce the same Overall Impact Score and Overall Risk Score within a BSR, although the intermediate scores describe different components of the overall score.


In [11]:
# Add the applicable population priority to each fish-use row. A left join
# preserves every fish-use row; later QC stops if any priority was not found.
fish_population = fish_use.merge(
    population[["basin", "species", "life_stage", "population_priority"]],
    on=["basin", "species", "life_stage"],
    how="left",
    validate="many_to_one",
)

# Expand each fish-use row across all limiting factors by joining the matching
# vulnerability rows for its species and life stage.
calculation_grid = fish_population.merge(
    vulnerability_collapsed,
    on=["species", "life_stage"],
    how="left",
    validate="many_to_many",
)

# Add the one condition record for the same BSR and limiting factor.
calculation_grid = calculation_grid.merge(
    condition[
        [
            "bsr", "limiting_factor", "condition_score_raw_1_5",
            "condition_score_source", "condition_score", "n", "any_flag",
        ]
    ],
    on=["bsr", "limiting_factor"],
    how="left",
    validate="many_to_one",
)

# Calculate life-stage fish use × vulnerability. This is summed in the
# limiting-factor impact equation.
calculation_grid["fish_vulnerability_component"] = (
    calculation_grid["LS_corrected_score"]
    * calculation_grid["vulnerability_score"]
)

# Calculate condition × vulnerability. This is summed in the life-stage impact
# equation.
calculation_grid["condition_vulnerability_component"] = (
    calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
)

# Calculate life-stage fish use × vulnerability × population priority. This is
# summed in the limiting-factor risk equation.
calculation_grid["population_weighted_fish_vulnerability_component"] = (
    calculation_grid["fish_vulnerability_component"]
    * calculation_grid["population_priority"]
)

# Pathway impact = life-stage fish use × condition × vulnerability.
calculation_grid["impact_component"] = (
    calculation_grid["fish_vulnerability_component"]
    * calculation_grid["condition_score"]
)

# Pathway risk = pathway impact × population priority.
calculation_grid["risk_component"] = (
    calculation_grid["impact_component"]
    * calculation_grid["population_priority"]
)

# Display fields needed to audit the first ten pathway calculations.
display(
    calculation_grid[
        [
            "bsr", "species", "life_stage", "limiting_factor",
            "LS_corrected_score", "species_aggregate_score",
            "fish_use_score", "population_priority",
            "condition_score_raw_1_5", "condition_score_source",
            "condition_score", "vulnerability_rank_min",
            "vulnerability_rank_max", "vulnerability_score",
            "impact_component", "risk_component",
        ]
    ].head(10)
)


,bsr,species,life_stage,limiting_factor,LS_corrected_score,species_aggregate_score,fish_use_score,population_priority,condition_score_raw_1_5,condition_score_source,condition_score,vulnerability_rank_min,vulnerability_rank_max,vulnerability_score,impact_component,risk_component
0,CC1,Chinook,Spawning,Altered Flow Timing,0.0,0.77,0.4162,0.12,4.31,0.84475,0.829225,10,10,0.363571,0.0,0.0
1,CC1,Chinook,Spawning,Altered Primary Productivity,0.0,0.77,0.4162,0.12,4.31,0.84475,0.829225,14,14,0.080714,0.0,0.0
2,CC1,Chinook,Spawning,Anthropogenic Barriers,0.0,0.77,0.4162,0.12,3.68,0.70300,0.673300,15,15,0.010000,0.0,0.0
3,CC1,Chinook,Spawning,Channel and Habitat Structure,0.0,0.77,0.4162,0.12,4.57,0.90325,0.893575,5,5,0.717143,0.0,0.0
4,CC1,Chinook,Spawning,Decreased Sediment Quantity,0.0,0.77,0.4162,0.12,2.95,0.53875,0.492625,3,3,0.858571,0.0,0.0
5,CC1,Chinook,Spawning,Decreased Water Quantity,0.0,0.77,0.4162,0.12,5.00,1.00000,1.000000,2,2,0.929286,0.0,0.0
6,CC1,Chinook,Spawning,Floodplain Condition,0.0,0.77,0.4162,0.12,4.78,0.95050,0.945550,6,6,0.646429,0.0,0.0
7,CC1,Chinook,Spawning,Increased Sediment Quantity,0.0,0.77,0.4162,0.12,4.51,0.88975,0.878725,11,11,0.292857,0.0,0.0
8,CC1,Chinook,Spawning,Non-Native Species Interactions and Competition,0.0,0.77,0.4162,0.12,4.64,0.91900,0.910900,12,12,0.222143,0.0,0.0
9,CC1,Chinook,Spawning,Predation,0.0,0.77,0.4162,0.12,4.22,0.82450,0.806950,9,9,0.434286,0.0,0.0


## 5. Aggregate Level 1 scores

### Species and life-stage scores

For a given BSR, species, and life stage, the Life-Stage Fish Use Score is constant across the 15 limiting factors and can be placed in front of the sum:

$$
\begin{aligned}
\text{life-stage impact score}
={}&
\text{life-stage fish use}
\times
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\Bigl(
\text{limiting-factor condition}
\times
\text{vulnerability}
\Bigr)
\end{aligned}
$$

The population priority is also constant for the applicable basin, species, and life stage:

$$
\begin{aligned}
\text{life-stage risk score}
={}&
\text{life-stage impact score}
\times
\text{population priority}
\end{aligned}
$$

The risk score for the highest-priority life stage is:

$$
\begin{aligned}
\text{risk score for highest priority life stage}
={}&
\max_{\substack{\text{all species and}\\\text{life stages}}}
\left(
\text{life-stage risk score}
\right)
\end{aligned}
$$

The **Highest Priority Life Stage** is the species and life-stage combination with the largest Life-Stage Risk Score within the BSR.

### Limiting-factor scores

For a given BSR and limiting factor, the Limiting-Factor Condition is constant across species and life stages and can be placed in front of the sum:

$$
\begin{aligned}
\text{limiting-factor impact score}
={}&
\text{limiting-factor condition}
\times
\sum_{\substack{\text{all species and}\\\text{life stages}}}
\Bigl(
\text{life-stage fish use}
\times
\text{vulnerability}
\Bigr)
\end{aligned}
$$

$$
\begin{aligned}
\text{limiting-factor risk score}
={}&
\text{limiting-factor condition}
\times
\sum_{\substack{\text{all species and}\\\text{life stages}}}
\Bigl(
\text{life-stage fish use}
\times
\text{vulnerability}
\times
\text{population priority}
\Bigr)
\end{aligned}
$$

Within these sums, life-stage fish use, vulnerability, and population priority are specific to the applicable species and life stage. Limiting-factor condition is the single condition score for the BSR and limiting factor.

The risk score for the highest-priority limiting factor is:

$$
\begin{aligned}
\text{risk score for highest priority limiting factor}
={}&
\max_{\text{15 limiting factors}}
\left(
\text{limiting-factor risk score}
\right)
\end{aligned}
$$

The **Highest Priority Limiting Factor** is the limiting factor with the largest Limiting-Factor Risk Score within the BSR.

### Overall BSR scores

The Overall Impact Score can be calculated equivalently by summing either all Life-Stage Impact Scores or all 15 Limiting-Factor Impact Scores:

$$
\begin{aligned}
\text{Overall Impact Score}
&=
\sum_{\substack{\text{all species and}\\\text{life stages}}}
\left(
\text{life-stage impact score}
\right) \\
&=
\sum_{\text{15 limiting factors}}
\left(
\text{limiting-factor impact score}
\right)
\end{aligned}
$$

The Overall Risk Score can likewise be calculated equivalently by summing either all Life-Stage Risk Scores or all 15 Limiting-Factor Risk Scores:

$$
\begin{aligned}
\text{Overall Risk Score}
&=
\sum_{\substack{\text{all species and}\\\text{life stages}}}
\left(
\text{life-stage risk score}
\right) \\
&=
\sum_{\text{15 limiting factors}}
\left(
\text{limiting-factor risk score}
\right)
\end{aligned}
$$


In [12]:
# Group pathway rows by BSR, species, and life stage.
life_stage_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "species", "life_stage"], as_index=False
    )
    .agg(
        # These values are constant within each group, so retain the first.
        LS_corrected_score=("LS_corrected_score", "first"),
        species_aggregate_score=("species_aggregate_score", "first"),
        fish_use_score=("fish_use_score", "first"),
        population_priority=("population_priority", "first"),

        # Sum condition × vulnerability across all 15 limiting factors.
        condition_vulnerability_sum=(
            "condition_vulnerability_component", "sum"
        ),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)

# Apply life-stage fish use to the summed condition/vulnerability contribution.
life_stage_scores["impact_score"] = (
    life_stage_scores["LS_corrected_score"]
    * life_stage_scores["condition_vulnerability_sum"]
)

# Apply population priority to the life-stage impact score.
life_stage_scores["risk_score"] = (
    life_stage_scores["impact_score"]
    * life_stage_scores["population_priority"]
)

# Rank life-stage risk from largest to smallest within each BSR.
life_stage_scores["risk_rank_within_bsr"] = (
    life_stage_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Combine species and life stage into one readable label.
life_stage_scores["species_life_stage_label"] = (
    life_stage_scores["species"] + " | " + life_stage_scores["life_stage"]
)

# Group pathway rows by BSR and limiting factor.
limiting_factor_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "limiting_factor"], as_index=False
    )
    .agg(
        # Retain contextual fields that are constant within each group.
        fish_use_score=("fish_use_score", "first"),
        condition_score_raw_1_5=("condition_score_raw_1_5", "first"),
        condition_score_source=("condition_score_source", "first"),
        condition_score=("condition_score", "first"),
        condition_rating_n=("n", "first"),
        condition_review_flag=("any_flag", "first"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),

        # Sum fish use × vulnerability across species and life stages.
        fish_vulnerability_sum=("fish_vulnerability_component", "sum"),

        # Sum fish use × vulnerability × population priority.
        population_weighted_fish_vulnerability_sum=(
            "population_weighted_fish_vulnerability_component", "sum"
        ),
    )
)

# Apply condition to the summed fish-use/vulnerability contribution.
limiting_factor_scores["impact_score"] = (
    limiting_factor_scores["condition_score"]
    * limiting_factor_scores["fish_vulnerability_sum"]
)

# Apply condition to the population-weighted contribution.
limiting_factor_scores["risk_score"] = (
    limiting_factor_scores["condition_score"]
    * limiting_factor_scores[
        "population_weighted_fish_vulnerability_sum"
    ]
)

# Rank limiting-factor risk from largest to smallest within each BSR.
limiting_factor_scores["risk_rank_within_bsr"] = (
    limiting_factor_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Reduce repeated BSR fish-use values to one contextual row per BSR.
source_fish_use = (
    fish_use.groupby(["bsr", "basin"], as_index=False)
    .agg(
        fish_use_score=("fish_use_score", "first"),
        fish_use_score_variants=("fish_use_score", "nunique"),
    )
)

# Sum all species and life-stage scores to obtain BSR totals.
bsr_from_life_stage = (
    life_stage_scores.groupby(["bsr", "basin"], as_index=False)
    .agg(
        overall_impact_score=("impact_score", "sum"),
        overall_risk_score=("risk_score", "sum"),
    )
)

# Independently sum all limiting-factor scores for the balance check.
bsr_from_limiting_factor = (
    limiting_factor_scores.groupby("bsr", as_index=False)
    .agg(
        lf_sum_impact_score=("impact_score", "sum"),
        lf_sum_risk_score=("risk_score", "sum"),
    )
)

# Keep all rank-1 life stages and summarize any ties.
top_life_stage = (
    life_stage_scores.loc[life_stage_scores["risk_rank_within_bsr"].eq(1)]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_species_life_stage=(
            "species_life_stage_label",
            lambda values: "; ".join(sorted(values)),
        ),
        top_species_life_stage_risk_score=("risk_score", "first"),
        top_species_life_stage_risk_tie_count=(
            "species_life_stage_label", "size"
        ),
    )
)

# Keep all rank-1 limiting factors and summarize any ties.
top_limiting_factor = (
    limiting_factor_scores.loc[
        limiting_factor_scores["risk_rank_within_bsr"].eq(1)
    ]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_limiting_factor=(
            "limiting_factor", lambda values: "; ".join(sorted(values))
        ),
        top_limiting_factor_risk_score=("risk_score", "first"),
        top_limiting_factor_risk_tie_count=("limiting_factor", "size"),
    )
)

# Join totals, contextual fish use, and highest-priority summaries.
bsr_scores = (
    bsr_from_life_stage
    .merge(bsr_from_limiting_factor, on="bsr", validate="one_to_one")
    .merge(source_fish_use, on=["bsr", "basin"], validate="one_to_one")
    .merge(top_life_stage, on="bsr", validate="one_to_one")
    .merge(top_limiting_factor, on="bsr", validate="one_to_one")
)

# Both differences should be zero except for floating-point rounding.
bsr_scores["impact_balance_difference"] = (
    bsr_scores["overall_impact_score"]
    - bsr_scores["lf_sum_impact_score"]
)
bsr_scores["risk_balance_difference"] = (
    bsr_scores["overall_risk_score"]
    - bsr_scores["lf_sum_risk_score"]
)

display(
    bsr_scores[
        [
            "bsr", "overall_risk_score",
            "highest_risk_species_life_stage",
            "highest_risk_limiting_factor",
            "fish_use_score",
        ]
    ].sort_values("overall_risk_score", ascending=False).head(10)
)


,bsr,overall_risk_score,highest_risk_species_life_stage,highest_risk_limiting_factor,fish_use_score
24,UGR5,18.457541,Chinook | Migration,Decreased Water Quantity,0.5242
22,UGR3,15.653272,Chinook | Migration,Predation,0.6514
0,CC1,13.568398,Chinook | Migration,Decreased Water Quantity,0.4162
26,UGR7,11.608004,Chinook | Migration,Channel and Habitat Structure,0.3375
5,CC6,10.020813,Bull Trout | FMO,Decreased Water Quantity,0.6329
17,UGR17,10.014447,Bull Trout | FMO,Channel and Habitat Structure,0.6039
4,CC5,9.324468,Bull Trout | FMO,Channel and Habitat Structure,0.4663
11,UGR11,9.090582,Chinook | Migration,Channel and Habitat Structure,0.4504
6,CC7,8.842589,Bull Trout | Spawning & Resident,Summer Water Temperature,1.0000
21,UGR20,8.795318,Bull Trout | Spawning & Resident,Non-Native Species Interactions and Competition,0.8697


## 6. Calculate Level 2 action-specific and overall scores

The action weight is calculated from relationship directness and frequency:

$$
\text{action weight}
=
\text{relationship directness}
\times
\text{frequency}
$$

For each action:

$$
\begin{aligned}
\text{condition improvement score}
={}&
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\left(
\text{limiting-factor condition}
\times
\text{action weight}
\right)
\end{aligned}
$$

$$
\begin{aligned}
\text{limiting-factor amelioration score}
={}&
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\left(
\text{limiting-factor impact score}
\times
\text{action weight}
\right)
\end{aligned}
$$

For each limiting factor and action:

$$
\text{action-specific benefit component}
=
\text{limiting-factor risk score}
\times
\text{action weight}
$$

For each action, the action-specific benefit components are summed across the 15 limiting factors:

$$
\begin{aligned}
\text{action-specific benefit score}
={}&
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\left(
\text{limiting-factor risk score}
\times
\text{action weight}
\right)
\end{aligned}
$$

The Overall Benefit Score is the sum of the Action-Specific Benefit Scores across all actions:

$$
\begin{aligned}
\text{overall benefit score}
={}&
\sum_{\text{all actions}}
\left(
\text{action-specific benefit score}
\right)
\end{aligned}
$$

The **Highest Risk-Aligned Action Type** is the action type with the largest Action-Specific Benefit Score within the BSR.


In [13]:
# Join each limiting-factor score to every action related to that factor.
# Each result is one BSR × limiting factor × action row.
action_components = limiting_factor_scores.merge(
    lfat[
        [
            "action_id", "action_type", "action_definition",
            "limiting_factor", "directness_code", "directness_value",
            "frequency_code", "frequency_value", "lfat_score",
        ]
    ],
    on="limiting_factor",
    how="inner",
    validate="many_to_many",
)

# Condition-improvement component = condition × action weight.
action_components["condition_improvement_component"] = (
    action_components["condition_score"]
    * action_components["lfat_score"]
)

# Amelioration component = limiting-factor impact × action weight.
action_components["amelioration_component"] = (
    action_components["impact_score"]
    * action_components["lfat_score"]
)

# Benefit component = limiting-factor risk × action weight.
action_components["benefit_component"] = (
    action_components["risk_score"]
    * action_components["lfat_score"]
)

# Sum all 15 limiting-factor contributions for each BSR and action.
action_scores = (
    action_components.groupby(
        [
            "bsr", "basin", "action_id", "action_type",
            "action_definition",
        ],
        as_index=False,
    )
    .agg(
        condition_improvement_score=(
            "condition_improvement_component", "sum"
        ),
        limiting_factor_amelioration_score=(
            "amelioration_component", "sum"
        ),
        action_benefit_score=("benefit_component", "sum"),

        # Count condition and vulnerability records flagged for review.
        condition_review_count=(
            "condition_review_flag",
            lambda values: int(
                values.fillna(False)
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["true", "yes", "1"])
                .sum()
            ),
        ),
        vulnerability_review_count=(
            "vulnerability_review_flag",
            lambda values: int(values.eq("Yes").sum()),
        ),
    )
)

# Add an action-specific display label.
action_scores["benefit_score_label"] = (
    action_scores["action_type"].astype(str) + " benefit score"
)

# Rank action benefit from largest to smallest within each BSR.
action_scores["benefit_rank_within_bsr"] = (
    action_scores.groupby("bsr")["action_benefit_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Keep all rank-1 actions and summarize any ties.
top_action = (
    action_scores.loc[action_scores["benefit_rank_within_bsr"].eq(1)]
    .assign(
        priority_action_label=lambda table: (
            table["action_id"].astype(str) + " | " + table["action_type"]
        )
    )
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_aligned_action_type=(
            "priority_action_label",
            lambda values: "; ".join(sorted(values)),
        ),
        highest_action_benefit_score=("action_benefit_score", "first"),
        top_action_benefit_tie_count=("priority_action_label", "size"),
    )
)

# Sum action-specific values. Overall Benefit Score is the HTML-defined total;
# the other two fields are retained as supplemental alignment summaries.
overall_action_scores = (
    action_scores.groupby("bsr", as_index=False)
    .agg(
        overall_condition_improvement_score=(
            "condition_improvement_score", "sum"
        ),
        overall_limiting_factor_amelioration_score=(
            "limiting_factor_amelioration_score", "sum"
        ),
        overall_benefit_score=("action_benefit_score", "sum"),
    )
)

# Add action summaries to the BSR output.
bsr_scores = (
    bsr_scores
    .merge(top_action, on="bsr", validate="one_to_one")
    .merge(overall_action_scores, on="bsr", validate="one_to_one")
)

display(
    action_scores[
        [
            "bsr", "action_type", "benefit_score_label",
            "condition_improvement_score",
            "limiting_factor_amelioration_score", "action_benefit_score",
            "benefit_rank_within_bsr",
        ]
    ].sort_values(["bsr", "benefit_rank_within_bsr"]).head(15)
)

display(
    bsr_scores[
        [
            "bsr", "overall_condition_improvement_score",
            "overall_limiting_factor_amelioration_score",
            "overall_benefit_score",
        ]
    ].sort_values("overall_benefit_score", ascending=False).head(10)
)


,bsr,action_type,benefit_score_label,condition_improvement_score,limiting_factor_amelioration_score,action_benefit_score,benefit_rank_within_bsr
2,CC1,Floodplain – Reconnect and Restore,Floodplain – Reconnect and Restore benefit score,5.546241,19.394964,5.870404,1
3,CC1,Riparian Vegetation Restoration,Riparian Vegetation Restoration benefit score,4.997469,16.896559,5.130133,2
5,CC1,Instream Flow Restoration,Instream Flow Restoration benefit score,3.895191,15.499464,4.732820,3
1,CC1,Instream Complexity Improvement,Instream Complexity Improvement benefit score,3.840793,14.728570,4.460576,4
0,CC1,Protect Land (Easement and Acquisition),Protect Land (Easement and Acquisition) benefi...,2.693367,9.948305,2.998294,5
4,CC1,Fish Passage – Barrier Removal and Replacement,Fish Passage – Barrier Removal and Replacement...,1.909181,7.686999,2.308642,6
7,CC1,Water Quality Improvement (not including tempe...,Water Quality Improvement (not including tempe...,1.767955,6.596781,1.994726,7
6,CC1,"Thermal Refuge Enhancement (reconnect, expand)","Thermal Refuge Enhancement (reconnect, expand)...",2.045837,6.287373,1.969576,8
10,CC1,Species Management (non-native or unnatural),Species Management (non-native or unnatural) b...,1.826342,6.159804,1.836585,9
9,CC1,Upland Treatments,Upland Treatments benefit score,1.551439,5.101809,1.544149,10


,bsr,overall_condition_improvement_score,overall_limiting_factor_amelioration_score,overall_benefit_score
24,UGR5,30.764902,148.852250,45.677857
22,UGR3,20.781198,116.233158,36.288199
0,CC1,31.272698,111.277380,33.737015
26,UGR7,31.361650,95.574035,29.367053
17,UGR17,20.922372,109.773427,27.446615
5,CC6,18.397501,103.796303,25.770844
4,CC5,21.811788,87.513889,23.742201
11,UGR11,21.330227,77.012584,23.598024
6,CC7,10.466908,75.035324,22.982225
15,UGR15,15.893717,85.274801,21.002822


## 7. Save core outputs, QC tables, and the scored GeoPackage

The seven core CSVs and scored GeoPackage are saved directly in `data/outputs`. The fish-use and population input tables are written both as CSVs and as nonspatial attribute tables in the GeoPackage. Supporting calculation components and review tables are saved in `data/outputs/QC`. The spatial output is required and is created on every successful run.


In [14]:
identifier_review = (
    fish_use[["bsr", "basin", "bsr_crosswalk_status"]]
    .drop_duplicates()
    .sort_values(["basin", "bsr"])
    .reset_index(drop=True)
)

CORE_SCORE_FILES = {
    "bsr": "bsr_scores.csv",
    "fish_use": "fish_use_scores.csv",
    "population": "population_scores.csv",
    "life_stage": "life_stage_scores.csv",
    "limiting_factor": "limiting_factor_scores_integrated.csv",
    "action": "action_scores.csv",
    "grid": "calculation_grid.csv",
}

CORE_OUTPUTS = {
    CORE_SCORE_FILES["bsr"]: bsr_scores,
    CORE_SCORE_FILES["fish_use"]: fish_use,
    CORE_SCORE_FILES["population"]: population,
    CORE_SCORE_FILES["life_stage"]: life_stage_scores,
    CORE_SCORE_FILES["limiting_factor"]: limiting_factor_scores,
    CORE_SCORE_FILES["action"]: action_scores,
    CORE_SCORE_FILES["grid"]: calculation_grid,
}
QC_OUTPUTS = {
    "action_score_components.csv": action_components,
    "assumptions_for_review.csv": assumptions,
    "bsr_identifiers_for_review.csv": identifier_review,
    "vulnerability_scores_for_review.csv": vulnerability_collapsed,
}

for filename, table in CORE_OUTPUTS.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
for filename, table in QC_OUTPUTS.items():
    table.to_csv(QC_DIR / filename, index=False)

ALL_CSV_OUTPUTS = {**CORE_OUTPUTS, **QC_OUTPUTS}
OUTPUT_PATHS = {
    **{name: OUTPUT_DIR / name for name in CORE_OUTPUTS},
    **{name: QC_DIR / name for name in QC_OUTPUTS},
}
GPKG_ATTRIBUTE_TABLES = {
    "fish_use_scores": {
        "table": fish_use,
        "identifier": "Atlas fish-use scores",
        "description": (
            "Source overall, species, and life-stage fish-use scores"
        ),
    },
    "population_scores": {
        "table": population,
        "identifier": "Atlas population scores",
        "description": (
            "Source population-priority scores by basin, species, and life stage"
        ),
    },
    "life_stage_scores": {
        "table": life_stage_scores,
        "identifier": "Atlas life-stage scores",
        "description": (
            "Level 1 species and life-stage scores by BSR for attribute joins"
        ),
    },
    "limiting_factor_scores": {
        "table": limiting_factor_scores,
        "identifier": "Atlas limiting-factor scores",
        "description": (
            "Level 1 limiting-factor scores by BSR for attribute joins"
        ),
    },
    "action_type_scores": {
        "table": action_scores,
        "identifier": "Atlas action-type scores",
        "description": (
            "Level 2 action-type scores by BSR for attribute joins"
        ),
    },
}


def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def sqlite_type(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_numeric_dtype(series):
        return "REAL"
    return "TEXT"


def sqlite_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def geometry_digest(path, feature_table, key_field, geometry_field):
    digest = hashlib.sha256()
    query = (
        f"SELECT {quote_identifier(key_field)}, "
        f"{quote_identifier(geometry_field)} "
        f"FROM {quote_identifier(feature_table)} "
        f"ORDER BY {quote_identifier(key_field)}"
    )
    with closing(sqlite3.connect(path)) as connection:
        for key, geometry in connection.execute(query):
            digest.update(str(key).encode("utf-8"))
            digest.update(bytes(geometry) if geometry is not None else b"")
    return digest.hexdigest()


def write_attribute_table(
    connection, table_name, table, identifier, description
):
    columns = table.columns.tolist()
    normalized_columns = [str(column).lower() for column in columns]
    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            f"GeoPackage table {table_name} has duplicate column names."
        )
    if "fid" in normalized_columns:
        raise ValueError(
            f"GeoPackage table {table_name} already contains a fid field."
        )
    table_exists = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE name = ?", (table_name,)
    ).fetchone()
    if table_exists is not None:
        raise ValueError(
            f"GeoPackage already contains a table named {table_name}."
        )

    column_definitions = ["fid INTEGER PRIMARY KEY AUTOINCREMENT"]
    column_definitions.extend(
        f"{quote_identifier(column)} {sqlite_type(table[column])}"
        for column in columns
    )
    connection.execute(
        f"CREATE TABLE {quote_identifier(table_name)} "
        f"({', '.join(column_definitions)})"
    )

    placeholders = ", ".join("?" for _ in columns)
    insert_sql = (
        f"INSERT INTO {quote_identifier(table_name)} "
        f"({', '.join(quote_identifier(column) for column in columns)}) "
        f"VALUES ({placeholders})"
    )
    connection.executemany(
        insert_sql,
        (
            tuple(sqlite_value(value) for value in row)
            for row in table.itertuples(index=False, name=None)
        ),
    )
    connection.execute(
        "INSERT INTO gpkg_contents "
        "(table_name, data_type, identifier, description, last_change) "
        "VALUES (?, 'attributes', ?, ?, "
        "strftime('%Y-%m-%dT%H:%M:%fZ', 'now'))",
        (table_name, identifier, description),
    )
    if "bsr" in normalized_columns:
        bsr_column = columns[normalized_columns.index("bsr")]
        index_name = f"idx_{table_name}_bsr"
        connection.execute(
            f"CREATE INDEX {quote_identifier(index_name)} "
            f"ON {quote_identifier(table_name)} "
            f"({quote_identifier(bsr_column)})"
        )


def write_scored_bsr_gpkg(
    input_path, output_path, summary, attribute_tables
):
    # sqlite3 connections are closed explicitly because its context
    # manager commits or rolls back but does not close the file handle.
    # An open handle prevents Path.replace() on Windows.
    with closing(sqlite3.connect(input_path)) as connection:
        feature_tables = [
            row[0]
            for row in connection.execute(
                "SELECT table_name FROM gpkg_contents WHERE data_type = 'features'"
            )
        ]
        candidates = []
        for table_name in feature_tables:
            columns = [
                row[1]
                for row in connection.execute(
                    f"PRAGMA table_info({quote_identifier(table_name)})"
                )
            ]
            source_key = next(
                (column for column in columns if column.lower() == "bsr"),
                None,
            )
            if source_key is not None:
                candidates.append((table_name, source_key, columns))

        if len(candidates) != 1:
            raise ValueError(
                "Expected exactly one feature layer containing a BSR field; "
                f"found {len(candidates)}."
            )
        feature_table, source_key, source_columns = candidates[0]
        geometry_row = connection.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (feature_table,),
        ).fetchone()
        if geometry_row is None:
            raise ValueError(
                f"No geometry field is registered for layer {feature_table}."
            )
        geometry_field = geometry_row[0]
        spatial_keys = {
            str(row[0]).strip()
            for row in connection.execute(
                f"SELECT {quote_identifier(source_key)} "
                f"FROM {quote_identifier(feature_table)}"
            )
        }

    summary_keys = set(summary["bsr"].astype(str).str.strip())
    if spatial_keys != summary_keys:
        raise ValueError(
            "The GeoPackage and score summary have different BSR coverage. "
            f"Missing scores: {sorted(spatial_keys - summary_keys)}; "
            f"missing geometry: {sorted(summary_keys - spatial_keys)}"
        )

    scored = summary.rename(columns={"bsr": "score_bsr"}).copy()
    scored_columns = scored.columns.tolist()
    existing_lower = {column.lower() for column in source_columns}
    collisions = [
        column for column in scored_columns if column.lower() in existing_lower
    ]
    if collisions:
        raise ValueError(
            "Score fields collide with existing GeoPackage fields: "
            + ", ".join(collisions)
        )

    temporary_path = output_path.with_name(
        f".{output_path.stem}.{uuid4().hex}.tmp.gpkg"
    )
    shutil.copy2(input_path, temporary_path)

    try:
        with closing(sqlite3.connect(temporary_path)) as connection:
            # Some GeoPackages contain RTree update triggers that invoke these
            # functions even when only non-geometry fields are updated.
            connection.create_function(
                "ST_IsEmpty", 1, lambda geometry: 1 if geometry is None else 0
            )
            for function_name in ("ST_MinX", "ST_MaxX", "ST_MinY", "ST_MaxY"):
                connection.create_function(
                    function_name, 1, lambda geometry: 0.0
                )

            with connection:
                for column in scored_columns:
                    connection.execute(
                        f"ALTER TABLE {quote_identifier(feature_table)} "
                        f"ADD COLUMN {quote_identifier(column)} "
                        f"{sqlite_type(scored[column])}"
                    )

                assignments = ", ".join(
                    f"{quote_identifier(column)} = ?"
                    for column in scored_columns
                )
                update_sql = (
                    f"UPDATE {quote_identifier(feature_table)} "
                    f"SET {assignments} "
                    f"WHERE TRIM(CAST("
                    f"{quote_identifier(source_key)} AS TEXT)) = ?"
                )
                for original_bsr, (_, row) in zip(
                    summary["bsr"].astype(str), scored.iterrows()
                ):
                    values = [
                        sqlite_value(row[column])
                        for column in scored_columns
                    ]
                    values.append(original_bsr.strip())
                    cursor = connection.execute(update_sql, values)
                    if cursor.rowcount != 1:
                        raise ValueError(
                            f"Expected one spatial row for {original_bsr}; "
                            f"updated {cursor.rowcount}."
                        )

                connection.execute(
                    "UPDATE gpkg_contents "
                    "SET identifier = ?, description = ? "
                    "WHERE table_name = ?",
                    (
                        "Atlas scored BSRs",
                        "BSR geometry with Level 1 and Level 2 score summaries",
                        feature_table,
                    ),
                )

                for table_name, table_specification in attribute_tables.items():
                    write_attribute_table(
                        connection=connection,
                        table_name=table_name,
                        **table_specification,
                    )

            integrity = connection.execute(
                "PRAGMA integrity_check"
            ).fetchone()[0]
            if integrity != "ok":
                raise ValueError(
                    f"GeoPackage integrity check failed: {integrity}"
                )

        # Path.replace() overwrites an existing output atomically. The SQLite
        # connection must be closed before this line on Windows.
        try:
            temporary_path.replace(output_path)
        except PermissionError as error:
            raise PermissionError(
                f"Could not replace {output_path}. Close the output "
                "GeoPackage in QGIS, ArcGIS, or another application, then "
                "run this cell again."
            ) from error
    finally:
        # Do not let cleanup of a staging file hide the original exception.
        try:
            temporary_path.unlink(missing_ok=True)
        except OSError:
            pass

    return (
        feature_table, geometry_field, source_key, scored_columns,
        list(attribute_tables),
    )


(
    BSR_OUTPUT_LAYER,
    BSR_GEOMETRY_FIELD,
    BSR_SOURCE_KEY,
    BSR_SCORE_FIELDS,
    BSR_ATTRIBUTE_TABLES,
) = write_scored_bsr_gpkg(
    BSR_INPUT_PATH, BSR_OUTPUT_PATH, bsr_scores, GPKG_ATTRIBUTE_TABLES
)

manifest_rows = [
    {"file": filename, "rows": len(table), "path": OUTPUT_PATHS[filename]}
    for filename, table in ALL_CSV_OUTPUTS.items()
]
manifest_rows.append(
    {
        "file": BSR_OUTPUT_FILE,
        "rows": len(bsr_scores),
        "path": BSR_OUTPUT_PATH,
    }
)
output_manifest = pd.DataFrame(manifest_rows)
display(output_manifest)


,file,rows,path
0,bsr_scores.csv,29,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
1,fish_use_scores.csv,290,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
2,population_scores.csv,20,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
3,life_stage_scores.csv,290,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
4,limiting_factor_scores_integrated.csv,435,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
5,action_scores.csv,319,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
6,calculation_grid.csv,4350,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
7,action_score_components.csv,4785,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
8,assumptions_for_review.csv,9,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...
9,bsr_identifiers_for_review.csv,29,C:\Users\AlexThornton-Dunwood\OneDrive - Liche...


## 8. Interpretations and notes

- The Life-Stage Fish Use Score is retained on its source scale and is not constrained to 0 to 1.
- The BSR-level and species-level fish-use scores are context fields, not direct impact or risk multipliers.
- Vulnerability and population priority are specific to the applicable species and life stage.
- Limiting-factor condition is specific to the BSR and limiting factor. The same condition value is applied to all species and life-stage pathways for that factor within the BSR.
- Aggregate scores are sums and can exceed 1.
- `action_benefit_score` is action-specific. Its display label should identify the action type.
- Summing across action types can count a limiting-factor pathway more than once. Treat these as alignment scores, not additive estimates of realized benefit.
- Action scores do not account for feasibility, cost, landowner willingness, implementation constraints, or site-specific effectiveness.
- Combined migration uses the maximum adult/juvenile vulnerability; the review table retains the source stages and score range.


## 9. Quality control

The checks below verify schemas, 0.01-to-1.0 transformations, keys, 15-factor coverage, joins, pathway equations, agreement between aggregation routes, saved outputs, GeoPackage integrity, and geometry preservation. An assertion stops the notebook at the first failed requirement.


In [15]:
# QC 1: source values and score transformations

# Collect all required fields so missing values can be reported by name.
required_non_null = {
    "fish-use BSR": fish_use["bsr"],
    "life-stage fish-use score": fish_use["LS_corrected_score"],
    "species fish-use score": fish_use["species_aggregate_score"],
    "BSR fish-use score": fish_use["fish_use_score"],
    "population priority": population["population_priority"],
    "condition raw score": condition["condition_score_raw_1_5"],
    "condition source score": condition["condition_score_source"],
    "condition calculated score": condition["condition_score"],
    "vulnerability rank": vulnerability["vulnerability_rank"],
    "vulnerability source score": vulnerability["vulnerability_score_source"],
    "vulnerability calculated score": vulnerability["vulnerability_score"],
    "LFAT directness": lfat["directness_value"],
    "LFAT frequency": lfat["frequency_value"],
    "LFAT score": lfat["lfat_score"],
}
unresolved = {
    name: int(series.isna().sum())
    for name, series in required_non_null.items()
}
assert not any(unresolved.values()), unresolved

# Confirm input and multiplier ranges.
assert fish_use["fish_use_score"].between(0, 1).all()
assert fish_use["LS_corrected_score"].ge(0).all()
assert fish_use["species_aggregate_score"].ge(0).all()
assert population["population_priority"].between(0, 1).all()

# Confirm population priorities sum to approximately 1 by basin and species.
population_sums = (
    population.groupby(["basin", "species"])["population_priority"].sum()
)
assert np.allclose(population_sums, 1.0, atol=0.011)

# Independently reproduce and check the condition transformation.
expected_condition_score = (
    0.01
    + (condition["condition_score_raw_1_5"] - 1.0) * (0.99 / 4.0)
)
assert condition["condition_score_raw_1_5"].between(1, 5).all()
assert condition["condition_score"].between(0.01, 1.00).all()
assert np.allclose(
    condition["condition_score"], expected_condition_score, atol=1e-12
)

# Independently reproduce and check the vulnerability transformation.
expected_vulnerability_score = (
    1.0
    - (vulnerability["vulnerability_rank"] - 1.0) * (0.99 / 14.0)
)
assert vulnerability["vulnerability_rank"].between(1, 15).all()
assert vulnerability["vulnerability_score"].between(0.01, 1.00).all()
assert np.allclose(
    vulnerability["vulnerability_score"],
    expected_vulnerability_score,
    atol=1e-12,
)

# Each source life stage must include all 15 limiting factors.
vulnerability_source_coverage = vulnerability.groupby(
    ["species", "source_life_stage"]
).agg(
    rows=("limiting_factor", "size"),
    limiting_factors=("limiting_factor", "nunique"),
)
assert vulnerability_source_coverage["rows"].eq(15).all()
assert vulnerability_source_coverage["limiting_factors"].eq(15).all()

# Check that action weight equals directness × frequency.
assert np.allclose(
    lfat["lfat_score"],
    lfat["directness_value"] * lfat["frequency_value"],
    atol=1e-12,
)
assert lfat["lfat_score"].between(0, 1).all()

display(population_sums.rename("priority_sum").reset_index())
display(pd.DataFrame([unresolved]).T.rename(columns={0: "missing_values"}))


,basin,species,priority_sum
0,Catherine Creek,Bull Trout,1.00
1,Catherine Creek,Chinook,0.99
2,Catherine Creek,Steelhead,1.00
3,Upper Grande Ronde,Bull Trout,1.00
4,Upper Grande Ronde,Chinook,1.00
5,Upper Grande Ronde,Steelhead,1.00


,missing_values
fish-use BSR,0
life-stage fish-use score,0
species fish-use score,0
BSR fish-use score,0
population priority,0
condition raw score,0
condition source score,0
condition calculated score,0
vulnerability rank,0
vulnerability source score,0


In [16]:
# QC 2: keys, coverage, joins, balances, and saved outputs
assert not fish_use.duplicated(["bsr", "species", "life_stage"]).any()
assert not population.duplicated(["basin", "species", "life_stage"]).any()
assert not condition.duplicated(["bsr", "limiting_factor"]).any()
assert not vulnerability.duplicated(
    ["species", "source_life_stage", "limiting_factor"]
).any()
assert not lfat.duplicated(["action_id", "limiting_factor"]).any()
assert not vulnerability_collapsed.duplicated(
    ["species", "life_stage", "limiting_factor"]
).any()

canonical_lf_set = set(CANONICAL_LF)
assert set(condition["limiting_factor"]) == canonical_lf_set
assert set(vulnerability["limiting_factor"]) == canonical_lf_set
assert set(lfat["limiting_factor"]) == canonical_lf_set
assert set(fish_use["bsr"]) == set(condition["bsr"])
assert condition.groupby("bsr")["limiting_factor"].nunique().eq(15).all()
assert lfat.groupby("action_id")["limiting_factor"].nunique().eq(15).all()

fish_population_keys = set(
    map(
        tuple,
        fish_use[["basin", "species", "life_stage"]]
        .drop_duplicates()
        .to_numpy(),
    )
)
population_keys = set(
    map(tuple, population[["basin", "species", "life_stage"]].to_numpy())
)
assert fish_population_keys == population_keys

fish_stage_keys = set(
    map(
        tuple,
        fish_use[["species", "life_stage"]].drop_duplicates().to_numpy(),
    )
)
vulnerability_stage_keys = set(
    map(
        tuple,
        vulnerability_collapsed[["species", "life_stage"]]
        .drop_duplicates()
        .to_numpy(),
    )
)
assert fish_stage_keys == vulnerability_stage_keys

expected_vulnerability_rows = len(fish_stage_keys) * len(CANONICAL_LF)
expected_grid_rows = len(fish_use) * len(CANONICAL_LF)
required_grid_fields = [
    "LS_corrected_score", "species_aggregate_score",
    "fish_use_score", "population_priority", "vulnerability_score",
    "condition_score", "impact_component", "risk_component",
]
assert len(vulnerability_collapsed) == expected_vulnerability_rows
assert len(calculation_grid) == expected_grid_rows
assert not calculation_grid[required_grid_fields].isna().any().any()
assert calculation_grid.groupby("bsr").size().eq(
    len(fish_stage_keys) * len(CANONICAL_LF)
).all()
assert source_fish_use["fish_use_score_variants"].eq(1).all()
assert fish_use.groupby(["bsr", "species"])[
    "species_aggregate_score"
].nunique().eq(1).all()
score_keys = ["bsr", "basin", "species", "life_stage"]
source_detail_scores = fish_use.set_index(score_keys)[
    ["LS_corrected_score", "species_aggregate_score"]
].sort_index()
output_detail_scores = life_stage_scores.set_index(score_keys)[
    ["LS_corrected_score", "species_aggregate_score"]
].sort_index()
pd.testing.assert_frame_equal(output_detail_scores, source_detail_scores)
assert np.allclose(
    calculation_grid["impact_component"],
    calculation_grid["LS_corrected_score"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"],
)
assert np.allclose(
    calculation_grid["risk_component"],
    calculation_grid["impact_component"]
    * calculation_grid["population_priority"],
)
assert np.allclose(
    calculation_grid.loc[
        calculation_grid["LS_corrected_score"].eq(0), "impact_component"
    ],
    0.0,
)
assert np.allclose(
    calculation_grid.loc[
        calculation_grid["LS_corrected_score"].eq(0), "risk_component"
    ],
    0.0,
)
assert np.allclose(
    life_stage_scores.loc[
        life_stage_scores["LS_corrected_score"].eq(0),
        "risk_score",
    ],
    0.0,
)
top_life_stage_rows = life_stage_scores.loc[
    life_stage_scores["risk_rank_within_bsr"].eq(1)
]
positive_risk_bsrs = set(
    life_stage_scores.groupby("bsr")["risk_score"]
    .max()
    .loc[lambda values: values > 0]
    .index
)
assert top_life_stage_rows.loc[
    top_life_stage_rows["bsr"].isin(positive_risk_bsrs),
    "LS_corrected_score",
].gt(0).all()
assert np.allclose(bsr_scores["impact_balance_difference"], 0.0)
assert np.allclose(bsr_scores["risk_balance_difference"], 0.0)

# Confirm that each overall BSR action score is exactly the sum of its
# corresponding action-specific scores.
action_score_sums = (
    action_scores.groupby("bsr")
    .agg(
        overall_condition_improvement_score=(
            "condition_improvement_score", "sum"
        ),
        overall_limiting_factor_amelioration_score=(
            "limiting_factor_amelioration_score", "sum"
        ),
        overall_benefit_score=("action_benefit_score", "sum"),
    )
    .sort_index()
)
bsr_action_totals = (
    bsr_scores.set_index("bsr")[
        [
            "overall_condition_improvement_score",
            "overall_limiting_factor_amelioration_score",
            "overall_benefit_score",
        ]
    ]
    .sort_index()
)
pd.testing.assert_frame_equal(
    bsr_action_totals,
    action_score_sums,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)
assert all(path.exists() for path in OUTPUT_PATHS.values())
for filename, source_table in {
    CORE_SCORE_FILES["fish_use"]: fish_use,
    CORE_SCORE_FILES["population"]: population,
}.items():
    pd.testing.assert_frame_equal(
        pd.read_csv(OUTPUT_PATHS[filename]),
        source_table,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )
expected_gpkg_attribute_rows = {
    name: len(specification["table"])
    for name, specification in GPKG_ATTRIBUTE_TABLES.items()
}

qc_rows = [
    ["Fish-use BSRs", fish_use["bsr"].nunique(), condition["bsr"].nunique()],
    ["Condition BSRs", condition["bsr"].nunique(), fish_use["bsr"].nunique()],
    ["Canonical limiting factors", len(CANONICAL_LF), 15],
    ["Collapsed vulnerability relationships", len(vulnerability_collapsed), expected_vulnerability_rows],
    ["Level 1 calculation-grid rows", len(calculation_grid), expected_grid_rows],
    ["LFAT action-factor relationships", len(lfat), lfat["action_id"].nunique() * len(CANONICAL_LF)],
    ["Saved CSV tables", sum(path.exists() for path in OUTPUT_PATHS.values()), len(OUTPUT_PATHS)],
]

assert BSR_OUTPUT_PATH.exists()
with closing(sqlite3.connect(BSR_OUTPUT_PATH)) as connection:
    scored_row_count = connection.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(BSR_OUTPUT_LAYER)}"
    ).fetchone()[0]
    scored_null_count = connection.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(BSR_OUTPUT_LAYER)} "
        "WHERE score_bsr IS NULL OR overall_risk_score IS NULL "
        f"OR {quote_identifier(BSR_GEOMETRY_FIELD)} IS NULL"
    ).fetchone()[0]
    gpkg_attribute_rows = {
        table_name: connection.execute(
            f"SELECT COUNT(*) FROM {quote_identifier(table_name)}"
        ).fetchone()[0]
        for table_name in BSR_ATTRIBUTE_TABLES
    }
    registered_attribute_tables = {
        row[0]
        for row in connection.execute(
            "SELECT table_name FROM gpkg_contents "
            "WHERE data_type = 'attributes'"
        )
    }
    scored_integrity = connection.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

assert scored_row_count == len(bsr_scores)
assert scored_null_count == 0
assert gpkg_attribute_rows == expected_gpkg_attribute_rows
assert set(BSR_ATTRIBUTE_TABLES).issubset(registered_attribute_tables)
assert scored_integrity == "ok"
assert geometry_digest(
    BSR_INPUT_PATH,
    BSR_OUTPUT_LAYER,
    BSR_SOURCE_KEY,
    BSR_GEOMETRY_FIELD,
) == geometry_digest(
    BSR_OUTPUT_PATH,
    BSR_OUTPUT_LAYER,
    BSR_SOURCE_KEY,
    BSR_GEOMETRY_FIELD,
)
qc_rows.extend(
    [
        ["Scored GeoPackage BSR rows", scored_row_count, len(bsr_scores)],
        ["Scored GeoPackage null score/geometry rows", scored_null_count, 0],
        *[
            [
                f"GeoPackage {table_name} rows",
                gpkg_attribute_rows[table_name],
                expected_gpkg_attribute_rows[table_name],
            ]
            for table_name in BSR_ATTRIBUTE_TABLES
        ],
    ]
)

qc_summary = pd.DataFrame(
    qc_rows, columns=["check", "observed", "expected"]
)
qc_summary["pass"] = qc_summary["observed"].eq(qc_summary["expected"])
assert qc_summary["pass"].all()

display(qc_summary)
display(
    identifier_review.query("bsr_crosswalk_status != 'exact_identifier'")
)


,check,observed,expected,pass
0,Fish-use BSRs,29,29,True
1,Condition BSRs,29,29,True
2,Canonical limiting factors,15,15,True
3,Collapsed vulnerability relationships,150,150,True
4,Level 1 calculation-grid rows,4350,4350,True
5,LFAT action-factor relationships,165,165,True
6,Saved CSV tables,11,11,True
7,Scored GeoPackage BSR rows,29,29,True
8,Scored GeoPackage null score/geometry rows,0,0,True
9,GeoPackage fish_use_scores rows,290,290,True


,bsr,basin,bsr_crosswalk_status
0,CC1,Catherine Creek,provisional_positional
1,CC2,Catherine Creek,provisional_positional
2,CC3,Catherine Creek,provisional_positional
3,CC4,Catherine Creek,provisional_positional
4,CC5,Catherine Creek,provisional_positional
5,CC6,Catherine Creek,provisional_positional
6,CC7,Catherine Creek,provisional_positional
7,CC8,Catherine Creek,provisional_positional
8,CC9,Catherine Creek,provisional_positional


### QC 3. Hand-calculated example

This example verifies that life-stage fish use is the impact multiplier, risk equals impact × population priority, both aggregation routes balance, and zero life-stage fish use produces zero impact and risk. The BSR fish-use value is retained only as context.


In [17]:
# Define two life stages and their population priorities.
test_population = {"Spawning": 0.60, "Rearing": 0.40}

# This BSR fish-use value is contextual and is not used below.
test_fish_use_score = 0.75

# These life-stage scores are the fish-use multipliers.
test_life_stage_fish_use = {"Spawning": 0.00, "Rearing": 0.80}

# Define two condition scores and pathway-specific vulnerabilities.
test_condition = {"Temperature": 0.10, "Instream Complexity": 1.00}
test_vulnerability = {
    ("Spawning", "Temperature"): 1.00,
    ("Rearing", "Temperature"): 0.50,
    ("Spawning", "Instream Complexity"): 0.50,
    ("Rearing", "Instream Complexity"): 1.00,
}
test_action_weight = {
    ("Floodplain Restoration", "Temperature"): 0.50,
    ("Floodplain Restoration", "Instream Complexity"): 1.00,
    ("Riparian Planting", "Temperature"): 1.00,
    ("Riparian Planting", "Instream Complexity"): 0.50,
}

# Calculate one impact and risk component for each pathway.
test_rows = []
for life_stage in test_population:
    for limiting_factor, condition_score in test_condition.items():
        # Impact = life-stage fish use × condition × vulnerability.
        impact = (
            test_life_stage_fish_use[life_stage]
            * condition_score
            * test_vulnerability[(life_stage, limiting_factor)]
        )

        # Risk = impact × population priority.
        risk = impact * test_population[life_stage]

        # Store each pathway so it can be aggregated in two directions.
        test_rows.append(
            {
                "life_stage": life_stage,
                "limiting_factor": limiting_factor,
                "fish_use_score": test_fish_use_score,
                "LS_corrected_score": test_life_stage_fish_use[life_stage],
                "condition_score": condition_score,
                "impact_component": impact,
                "risk_component": risk,
            }
        )
test_grid = pd.DataFrame(test_rows)

# Sum by life stage and independently by limiting factor.
test_life = test_grid.groupby("life_stage", as_index=False).agg(
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)
test_lf = test_grid.groupby("limiting_factor", as_index=False).agg(
    condition_score=("condition_score", "first"),
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)

# Apply action weights to condition, impact, and risk.
test_actions = []
for action in ["Floodplain Restoration", "Riparian Planting"]:
    rows = test_lf.copy()
    rows["weight"] = rows["limiting_factor"].map(
        lambda factor: test_action_weight[(action, factor)]
    )
    test_actions.append(
        {
            "action": action,
            "benefit_score_label": f"{action} benefit score",
            "condition_improvement_score": (
                rows["condition_score"] * rows["weight"]
            ).sum(),
            "limiting_factor_amelioration_score": (
                rows["impact_score"] * rows["weight"]
            ).sum(),
            "action_benefit_score": (
                rows["risk_score"] * rows["weight"]
            ).sum(),
        }
    )
test_actions = pd.DataFrame(test_actions)

# Sum the action-specific scores across actions.
test_overall_action_scores = pd.Series(
    {
        "overall_condition_improvement_score": (
            test_actions["condition_improvement_score"].sum()
        ),
        "overall_limiting_factor_amelioration_score": (
            test_actions["limiting_factor_amelioration_score"].sum()
        ),
        "overall_benefit_score": test_actions["action_benefit_score"].sum(),
    }
)

# Check the life-stage results.
expected_life = {
    "Spawning": (0.00, 0.00),
    "Rearing": (0.84, 0.336),
}
for row in test_life.itertuples(index=False):
    expected_impact, expected_risk = expected_life[row.life_stage]
    assert np.isclose(row.impact_score, expected_impact)
    assert np.isclose(row.risk_score, expected_risk)

# Check the contextual field, zero-score rule, and aggregation balance.
assert test_grid["fish_use_score"].eq(test_fish_use_score).all()
zero_rows = test_grid["LS_corrected_score"].eq(0)
assert test_grid.loc[zero_rows, "impact_component"].eq(0).all()
assert test_grid.loc[zero_rows, "risk_component"].eq(0).all()
assert np.isclose(test_life["impact_score"].sum(), 0.84)
assert np.isclose(test_life["risk_score"].sum(), 0.336)
assert np.isclose(
    test_life["impact_score"].sum(), test_lf["impact_score"].sum()
)
assert np.isclose(
    test_life["risk_score"].sum(), test_lf["risk_score"].sum()
)

# Check action-specific and overall results.
assert np.allclose(
    test_actions["condition_improvement_score"], [1.05, 0.60]
)
assert np.allclose(
    test_actions["limiting_factor_amelioration_score"], [0.82, 0.44]
)
assert np.allclose(test_actions["action_benefit_score"], [0.328, 0.176])
assert np.isclose(
    test_overall_action_scores["overall_condition_improvement_score"], 1.65
)
assert np.isclose(
    test_overall_action_scores["overall_limiting_factor_amelioration_score"],
    1.26,
)
assert np.isclose(
    test_overall_action_scores["overall_benefit_score"], 0.504
)

print("All schema, transformation, join, balance, output, and equation checks passed.")
display(test_life)
display(test_lf)
display(test_actions)
display(test_overall_action_scores.to_frame("score"))


All schema, transformation, join, balance, output, and equation checks passed.


,life_stage,impact_score,risk_score
0,Rearing,0.84,0.336
1,Spawning,0.00,0.000


,limiting_factor,condition_score,impact_score,risk_score
0,Instream Complexity,1.0,0.80,0.320
1,Temperature,0.1,0.04,0.016


,action,benefit_score_label,condition_improvement_score,limiting_factor_amelioration_score,action_benefit_score
0,Floodplain Restoration,Floodplain Restoration benefit score,1.05,0.82,0.328
1,Riparian Planting,Riparian Planting benefit score,0.60,0.44,0.176


,score
overall_condition_improvement_score,1.650
overall_limiting_factor_amelioration_score,1.260
overall_benefit_score,0.504
